In [ ]:
#| default_exp pipeline

In [ ]:
#| export
from __future__ import annotations
import asyncio, json, os, shlex, time
from shutil import which
from fastcore.all import L, Path, first
from fastcore.ansi import strip_ansi

from pullup.env import EnvStore, venv_env
from pullup.project import Step, default_steps, release_flow, app_project

In [ ]:
#| export
#: Bytes of transcript one pipeline keeps. A viewer that joins late is primed with this much.
SCROLLBACK = 200_000
#: Where a pipeline's plan is written, relative to the project. A caller may name another.
DIR = '.pullup'

class PipelineError(RuntimeError): pass

In [ ]:
#| export
def resolve_script(argv, python=None, env=None):
    "The executable to spawn, accepting either spelling of a hyphenated command."
    name = argv[0]
    spellings = (name, name.replace('_', '-'), name.replace('-', '_'))
    bindir = str(Path(python).parent) if python else ''
    for path in (bindir, (env or os.environ).get('PATH')):
        if not path: continue
        for candidate in spellings:
            if hit := which(candidate, path=path): return [hit, *argv[1:]]
    return argv

def uv_argv(root, argv):
    """`argv` through `uv run` where uv owns the project, else None.

    uv syncs the lock before it runs anything, so a venv that has drifted is repaired rather than
    failing halfway through with an import error.
    """
    root = Path(root)
    if not (root/'uv.lock').exists(): return None
    exe = which('uv')
    if not exe: return None
    # Spelled the way the project spells it. uv runs the project's own script where there is one
    # and falls through to PATH where there is not, and PATH is where another interpreter's
    # differently-spelled copy of the same tool is waiting.
    name = argv[0]
    bindir = root/'.venv'/'bin'
    hit = first(n for n in (name, name.replace('_', '-'), name.replace('-', '_'))
                if (bindir/n).exists())
    return [exe, 'run', '--project', str(root), hit or name.replace('_', '-'), *argv[1:]]

In [ ]:
#| export
class Pipeline:
    "A named sequence of commands for one project: its steps, its state, and its terminal."
    FILE = 'pipeline.json'
    KIND = 'pipeline'
    @staticmethod
    def defaults(root): return []
    def __init__(self,
        root,             # the project the commands run in
        python=None,      # the interpreter whose environment they run in; None uses this process's
        env=None,         # an `EnvStore`, or anything with `.get(key)` and `.values(keys)`
        dir=DIR,          # where the plan is written, relative to `root`
    ):
        self.root = Path(root).expanduser().resolve()
        self.python, self.dir = python, dir
        self.env = env if env is not None else EnvStore()
        self.steps = self._load()
        self.results, self.active = {}, ''
        self.pty = self._pump = None
        self._subs, self._auto = set(), True
        self.log = bytearray()
    @property
    def path(self): return self.root/self.dir/self.FILE
    def _load(self):
        try: raw = json.loads(self.path.read_text(encoding='utf-8'))
        except (OSError, ValueError): return self.defaults(self.root)
        steps = raw.get('steps') if isinstance(raw, dict) else raw
        rows = [r for r in (steps or ()) if isinstance(r, dict) and r.get('cmd')]
        out = [Step(id=str(row.get('id') or f'step{i + 1}'),
            label=str(row.get('label') or row.get('id') or 'step'),
            cmd=str(row['cmd']), doc=str(row.get('doc') or ''),
            needs=[str(x) for x in (row.get('needs') or ())],
            argv=[str(x) for x in (row.get('argv') or ())],
            meta=dict(row.get('meta') or {})) for i, row in enumerate(rows)]
        return out or self.defaults(self.root)
    def save(self, steps):
        "Persist an edited pipeline. Refuses a plan with no runnable command in it."
        rows = [s for s in (steps or ()) if str((s or {}).get('cmd') or '').strip()]
        if not rows: raise PipelineError('a pipeline needs at least one step with a command')
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.write_text(json.dumps({'steps': rows}, indent=2) + '\n', encoding='utf-8')
        self.steps = self._load()
        return self.state()
    def reset_plan(self):
        "Back to the default pipeline for this kind of project, without deleting the file."
        self.steps = self.defaults(self.root)
        return self.state()
    def step(self, step_id):
        found = first(self.steps, lambda s: s.id == str(step_id))
        if found is None: raise PipelineError(f'unknown {self.KIND} step: {step_id}')
        return found
    def needs(self):
        "Every environment key the pipeline names, in the order the steps name them."
        return list(L(self.steps).attrgot('needs').concat().unique())
    def missing(self, step):
        "Keys this step names that the environment store cannot produce."
        try: return [k for k in step.needs if not self.env.get(k)]
        except Exception: return [k for k in step.needs if not os.environ.get(k)]
    def _env(self):
        "The child's environment: this process, the project's interpreter, then the store on top."
        env = venv_env(self.python)
        try: env.update({k: str(v) for k, v in self.env.values(self.needs()).items()})
        except Exception: pass
        env.setdefault('PYTHONUNBUFFERED', '1')
        return env
    def _row(self, s):
        "One step as a caller sees it: the plan, plus how the last run of it went."
        r = self.results.get(s.id) or {}
        return s.dict() | {
            'status': 'running' if s.id == self.active else (r.get('status') or 'pending'),
            'code': r.get('code'), 'started': r.get('started', 0), 'finished': r.get('finished', 0),
            'missing': self.missing(s)}
    def state(self):
        "Everything about this pipeline that a caller could want to draw or assert on."
        rows = [self._row(s) for s in self.steps]
        failed = first(rows, lambda r: r['status'] == 'failed')
        return {'root': str(self.root), 'kind': self.KIND, 'path': str(self.path),
            'configured': self.path.exists(), 'steps': rows, 'active': self.active,
            'running': bool(self.active), 'auto': self._auto,
            'failed': failed['id'] if failed else '',
            'done': bool(rows) and all(r['status'] in ('passed', 'skipped') for r in rows),
            'needs': self.needs()}
    def subscribe(self):
        "A queue of terminal frames for one viewer, primed with what has already scrolled by."
        q = asyncio.Queue()
        if self.log: q.put_nowait(bytes(self.log))
        self._subs.add(q)
        return q
    def unsubscribe(self, q): self._subs.discard(q)
    def _broadcast(self, data):
        self.log += data
        if len(self.log) > SCROLLBACK: del self.log[:-SCROLLBACK]
        for q in list(self._subs):
            try: q.put_nowait(data)
            except Exception: self._subs.discard(q)
    def write(self, data):
        "Keystrokes into the running step: twine asks for a password."
        if self.pty is not None and self.pty.alive: self.pty.write(data)
    def resize(self, cols, rows):
        if self.pty is not None: self.pty.resize(rows, cols)
    def tail(self, lines=80):
        "The end of the transcript as plain text, escapes gone and bare returns made newlines."
        text = strip_ansi(bytes(self.log).decode('utf-8', 'replace'))
        rows = text.replace('\r\n', '\n').replace('\r', '\n').rstrip('\n').split('\n')
        return '\n'.join(rows[-max(1, int(lines)):])
    def _resolve(self, argv, env):
        "The executable to spawn, from this pipeline's interpreter."
        return uv_argv(self.root, argv) or resolve_script(argv, self.python, env)
    async def start(self, step_id='', auto=True):
        "Run one step, then keep going while they pass. Refuses to start a second one."
        if self.active: raise PipelineError(f'{self.active} is still running')
        self._auto = bool(auto)
        step = self.step(step_id) if step_id else self._next()
        if step is None: raise PipelineError('every step has already run — reset to run them again')
        await self._run(step)
        return self.state()
    def _next(self):
        "The first step that has not passed or been skipped."
        done = ('passed', 'skipped')
        return first(self.steps, lambda s: (self.results.get(s.id) or {}).get('status') not in done)
    def _after(self, step):
        i = [s.id for s in self.steps].index(step.id)
        return self.steps[i + 1] if i + 1 < len(self.steps) else None
    async def _run(self, step):
        from ptymini.core import PtySession
        argv = list(step.argv) if step.argv else shlex.split(step.cmd)
        if not argv: raise PipelineError(f'{step.id} has no command to run')
        self.active = step.id
        self.results[step.id] = {'status': 'running', 'code': None,
            'started': int(time.time()), 'finished': 0}
        env = self._env()
        self._broadcast(f'\r\n\x1b[1;36m❯ {step.cmd}\x1b[0m\r\n'.encode())
        try:
            self.pty = PtySession(self._resolve(argv, env), cwd=str(self.root), env=env,
                                  rows=30, cols=100, buffer_bytes=SCROLLBACK)
        except OSError as e:
            self.active = ''
            self.results[step.id] = {'status': 'failed', 'code': None, 'started': int(time.time()),
                'finished': int(time.time()), 'error': str(e)}
            self._broadcast(f'\r\n\x1b[31m{step.cmd}: {e}\x1b[0m\r\n'.encode())
            return
        self._pump = asyncio.create_task(self._drain(step))
    async def _drain(self, step):
        "Pump the step's pty to every viewer, record how it ended, then decide what happens next."
        pty = self.pty
        try:
            async for chunk in pty.attach(from_start=False): self._broadcast(chunk)
        except Exception as e:
            self._broadcast(f'\r\n\x1b[31m{type(e).__name__}: {e}\x1b[0m\r\n'.encode())
        code = pty.exit_code
        ok = code == 0
        self.results[step.id] = {'status': 'passed' if ok else 'failed', 'code': code,
            'started': (self.results.get(step.id) or {}).get('started', 0),
            'finished': int(time.time())}
        self.active = ''
        note = '\x1b[32m✓ ' if ok else '\x1b[31m✗ '
        self._broadcast(f'\r\n{note}{step.label} exited {code}\x1b[0m\r\n'.encode())
        if ok and self._auto and (nxt := self._after(step)) is not None: await self._run(nxt)
    def stop(self):
        "Stop the pipeline where it is. The step is killed; what it already did stays done."
        import signal
        self._auto = False
        step_id, self.active = self.active, ''
        if self.pty is not None and self.pty.alive:
            try: self.pty.kill(signal.SIGTERM)
            except ProcessLookupError: pass
        if step_id:
            self.results[step_id] = (self.results.get(step_id) or {}) | {
                'status': 'failed', 'code': None, 'finished': int(time.time())}
            self._broadcast(f'\r\n\x1b[33m{step_id} stopped\x1b[0m\r\n'.encode())
        return self.state()
    def skip(self, step_id):
        "Mark a step done without running it."
        step = self.step(step_id)
        if self.active == step.id: raise PipelineError(f'{step.id} is still running')
        self.results[step.id] = {'status': 'skipped', 'code': None, 'started': 0,
                                 'finished': int(time.time())}
        return self.state()
    def reset(self):
        "Forget what ran. The commands are unchanged."
        if self.active: raise PipelineError(f'{self.active} is still running')
        self.results.clear()
        self.log.clear()
        self._broadcast(b'\x1b[2J\x1b[H')
        return self.state()
    def blame(self, checkouts=(), family=None):
        "Which of your own packages raised in the last failure, pointed at your checkout of it."
        from pullup.stack import blame as attribute
        if not any((self.results.get(s.id) or {}).get('status') == 'failed' for s in self.steps):
            return {}
        return attribute(self.tail(400), checkouts=checkouts, family=family)

In [ ]:
#| export
class Release(Pipeline):
    "The release pipeline: nbdev's commands, fastship's for a package that is not one, else plain."
    FILE = 'release.json'
    KIND = 'release'
    @staticmethod
    def defaults(root): return default_steps(root)
    def state(self):
        flow = release_flow(self.root)
        return super().state() | {'flow': flow, 'nbdev': flow == 'nbdev',
            'fastship': flow == 'fastship', 'app': app_project(self.root)}